In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/Datasets/PercolationRate', exist_ok=True)

In [ ]:
import os
if not os.path.exists('HGP-clusterer'):
    !git clone https://github.com/Ludwig-H/HGP-clusterer.git
%cd HGP-clusterer
!pip install .

In [ ]:
import numpy as np
import scipy.sparse as sp
import os
import pickle
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
from hgp_delaunay import HGPDelaunay
from numba import njit

In [ ]:
@njit
def simulate_robust_dbscan(times, types, nodes_u, nodes_v, N):
    parent = np.arange(N)
    rank = np.zeros(N, dtype=np.int32)
    size_rob = np.zeros(N, dtype=np.int32)
    size_dbs = np.zeros(N, dtype=np.int32)
    
    ans_rob = np.full(N + 1, np.nan)
    ans_dbs = np.full(N + 1, np.nan)
    max_rob = 0
    max_dbs = 0
    
    for i in range(len(times)):
        t = times[i]
        typ = types[i]
        u = nodes_u[i]
        
        curr = u
        while parent[curr] != curr:
            curr = parent[curr]
        ru = curr
        curr = u
        while curr != ru:
            nxt = parent[curr]
            parent[curr] = ru
            curr = nxt
            
        if typ == 0:
            size_dbs[ru] += 1
            if size_dbs[ru] > max_dbs:
                ans_dbs[max_dbs + 1 : size_dbs[ru] + 1] = t
                max_dbs = size_dbs[ru]
        elif typ == 1:
            size_rob[ru] += 1
            if size_rob[ru] > max_rob:
                ans_rob[max_rob + 1 : size_rob[ru] + 1] = t
                max_rob = size_rob[ru]
        else:
            v = nodes_v[i]
            curr = v
            while parent[curr] != curr:
                curr = parent[curr]
            rv = curr
            curr = v
            while curr != rv:
                nxt = parent[curr]
                parent[curr] = rv
                curr = nxt
                
            if ru != rv:
                if rank[ru] < rank[rv]:
                    tmp = ru
                    ru = rv
                    rv = tmp
                parent[rv] = ru
                if rank[ru] == rank[rv]:
                    rank[ru] += 1
                    
                size_rob[ru] += size_rob[rv]
                size_dbs[ru] += size_dbs[rv]
                
                if size_rob[ru] > max_rob:
                    ans_rob[max_rob + 1 : size_rob[ru] + 1] = t
                    max_rob = size_rob[ru]
                if size_dbs[ru] > max_dbs:
                    ans_dbs[max_dbs + 1 : size_dbs[ru] + 1] = t
                    max_dbs = size_dbs[ru]
                    
    return ans_rob[1:], ans_dbs[1:]

@njit
def compute_wu(real_u, real_v, del_w, r_core, w_u, target_u):
    for i in range(len(real_u)):
        u = real_u[i]
        v = real_v[i]
        w = del_w[i]
        
        val_v = max(r_core[v], w)
        if val_v < w_u[u]:
            w_u[u] = val_v
            target_u[u] = v
            
        val_u = max(r_core[u], w)
        if val_u < w_u[v]:
            w_u[v] = val_u
            target_u[v] = u

def compute_hgp_percolation(faces_unique, e_u, e_v, e_w, n_points):
    F = faces_unique.shape[0]
    order = np.argsort(e_w)
    e_u = e_u[order]
    e_v = e_v[order]
    e_w = e_w[order]
    
    parent = np.arange(F)
    def find(i):
        path = []
        while parent[i] != i:
            path.append(i)
            i = parent[i]
        for node in path:
            parent[node] = i
        return i
    
    components = [set(faces_unique[i]) for i in range(F)]
    max_size = len(faces_unique[0]) if F > 0 else 0
    ans = np.full(n_points + 1, np.nan)
    ans[1:max_size + 1] = 0.0
    
    for i in range(len(e_w)):
        u = e_u[i]
        v = e_v[i]
        w = e_w[i]
        
        ru = find(u)
        rv = find(v)
        
        if ru != rv:
            if len(components[ru]) < len(components[rv]):
                ru, rv = rv, ru
            components[ru].update(components[rv])
            components[rv] = None
            parent[rv] = ru
            
            new_size = len(components[ru])
            if new_size > max_size:
                ans[max_size + 1 : new_size + 1] = w
                max_size = new_size
                
    return ans[1:]

In [ ]:
# @title Parameters and Main Loop
K_list = [1, 2, 3, 4, 5] # @param
d = 2 # @param
n = 1000000 # @param
T = 100 # @param

save_path = f'/content/drive/MyDrive/Datasets/PercolationRate/percolation_results_d{d}.pkl'

if os.path.exists(save_path):
    with open(save_path, 'rb') as f:
        results = pickle.load(f)
    print(f"Loaded existing results.")
else:
    results = {'hgp': {k: [] for k in K_list}, 
               'robust': {k: [] for k in K_list}, 
               'dbscan': {k: [] for k in K_list}}

tests_done = len(results['hgp'].get(K_list[0], []))
print(f"{tests_done} tests already done. {max(0, T - tests_done)} remaining.")

for t_idx in tqdm(range(tests_done, T), desc="Tests"):
    X = np.random.uniform(0, n**(1/d), size=(n, d)).astype(np.float32)
    
    # Get Delaunay edges for Robust and DBSCAN from HGP K=1
    hgp1 = HGPDelaunay(K=1, expZ=1.0, verbose=False)
    faces1, eu1, ev1, ew1 = hgp1.fit_transform(X)
    real_u = faces1[eu1, 0].astype(np.int32)
    real_v = faces1[ev1, 0].astype(np.int32)
    del_w = ew1.astype(np.float32)
    
    for K in K_list:
        if K not in results['hgp']:
            results['hgp'][K] = []
            results['robust'][K] = []
            results['dbscan'][K] = []
            
        # 1. HGP Components
        hgp = HGPDelaunay(K=K, expZ=1.0, verbose=False)
        faces, eu, ev, ew = hgp.fit_transform(X)
        ans_hgp = compute_hgp_percolation(faces, eu, ev, ew, n)
        results['hgp'][K].append(ans_hgp)
        
        # 2. Nearest Neighbors for Core Distance
        nn = NearestNeighbors(n_neighbors=K+1)
        nn.fit(X)
        dists, _ = nn.kneighbors(X)
        r_core = (dists[:, K] / 2.0).astype(np.float32)
        
        # 3. Simulate DBSCAN and Robust Single-Linkage
        w_u = r_core.copy()
        target_u = np.arange(n, dtype=np.int32)
        compute_wu(real_u, real_v, del_w, r_core, w_u, target_u)
        
        w_cc = np.maximum(r_core[real_u], r_core[real_v])
        w_cc = np.maximum(w_cc, del_w)
        
        E = len(real_u)
        times = np.concatenate([w_u, r_core, w_cc])
        types = np.concatenate([np.zeros(n, dtype=np.int8), 
                                np.ones(n, dtype=np.int8), 
                                np.full(E, 2, dtype=np.int8)])
        nodes_u = np.concatenate([target_u, np.arange(n, dtype=np.int32), real_u])
        nodes_v = np.concatenate([np.zeros(n, dtype=np.int32), np.zeros(n, dtype=np.int32), real_v])
        
        order = np.argsort(times)
        times = times[order]
        types = types[order]
        nodes_u = nodes_u[order]
        nodes_v = nodes_v[order]
        
        ans_rob, ans_dbs = simulate_robust_dbscan(times, types, nodes_u, nodes_v, n)
        results['robust'][K].append(ans_rob)
        results['dbscan'][K].append(ans_dbs)
        
    with open(save_path, 'wb') as f:
        pickle.dump(results, f)

In [ ]:
# @title Analysis and Plotting
epsilon = 0.03
n_min = int(n * epsilon)
n_max = int(n * (1 - epsilon))

v_results = {'hgp': {}, 'robust': {}, 'dbscan': {}}

for K in K_list:
    if len(results['hgp'][K]) == 0: continue
    
    avg_hgp = np.nanmean(np.array(results['hgp'][K]), axis=0)
    avg_rob = np.nanmean(np.array(results['robust'][K]), axis=0)
    avg_dbs = np.nanmean(np.array(results['dbscan'][K]), axis=0)
    
    r_min_hgp = avg_hgp[n_min]
    r_max_hgp = avg_hgp[n_max]
    v_results['hgp'][K] = (r_min_hgp / r_max_hgp)**d if r_max_hgp > 0 else 0
    
    r_min_rob = avg_rob[n_min]
    r_max_rob = avg_rob[n_max]
    v_results['robust'][K] = (r_min_rob / r_max_rob)**d if r_max_rob > 0 else 0
    
    r_min_dbs = avg_dbs[n_min]
    r_max_dbs = avg_dbs[n_max]
    v_results['dbscan'][K] = (r_min_dbs / r_max_dbs)**d if r_max_dbs > 0 else 0

print(f"=== Percolation Rate v (d={d}) ===")
print(f"{'K':<5} | {'HGP':<10} | {'Robust SL':<10} | {'DBSCAN-core':<10}")
print("-" * 45)
for K in K_list:
    if K not in v_results['hgp']: continue
    print(f"{K:<5} | {v_results['hgp'][K]:<10.4f} | {v_results['robust'][K]:<10.4f} | {v_results['dbscan'][K]:<10.4f}")

def plot_curves(res_dict, name):
    plt.figure(figsize=(10, 6))
    x_axis = np.arange(1, n+1) / n
    for K in K_list:
        if K not in res_dict: continue
        avg_curve = np.nanmean(np.array(res_dict[K]), axis=0)
        plt.plot(avg_curve, x_axis, label=f'{name} K={K}')
    plt.title(f'{name} Percolation Probability in {d}D')
    plt.xlabel('Radius r')
    plt.ylabel('Fraction of points in giant component')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_curves(results['hgp'], 'HGP')
plot_curves(results['robust'], 'Robust SL')
plot_curves(results['dbscan'], 'DBSCAN-core')